## Spark/Findspark

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()


In [2]:
df = spark.sql("""select 'Sucesso total, estamos online!' as hello""")
df.show()

+--------------------+
|               hello|
+--------------------+
|Sucesso total, es...|
+--------------------+



In [3]:
# Import spark libraries
from pyspark.sql import Row, DataFrame
from pyspark.sql.types import StringType, StructType, StructField, IntegerType
from pyspark.sql.functions import col, expr, lit, substring, concat, concat_ws, when, coalesce
from pyspark.sql import functions as F  # for more sql functions
from functools import reduce
from pyspark.sql.functions import to_date
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import StringType, StructType, StructField

In [4]:
import requests

path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'owid-covid-data.csv'
csv_file = open(csv_file_name, 'wb')

csv_file.write(url_content)
csv_file.close()

df = spark.read.csv(csv_file_name, header=True, inferSchema=True)


# Data Manipulation using Spark

In [5]:
caminho = r"C:\Users\00157NLUC-BrenoR\pos_data_analytics\banklist.csv"


In [6]:
schema = StructType([
    StructField("Bank Name�", StringType(), True),
    StructField("City�", StringType(), True),
    StructField("State�", StringType(), True),
    StructField("Cert�", StringType(), True),
    StructField("Acquiring Institution�", StringType(), True),
    StructField("Closing Date�", StringType(), True),
    StructField("Fund", StringType(), True),
])

In [7]:
df = spark.read.csv(
    caminho,
    header=True,
    schema=schema,
    encoding="UTF-8"
)


In [8]:
# Renomeia as colunas
df_clean = (
    df
    .withColumnRenamed("Bank Name�", "bank_name")
    .withColumnRenamed("City�", "city")
    .withColumnRenamed("State�", "state")
    .withColumnRenamed("Cert�", "cert")
    .withColumnRenamed("Acquiring Institution�", "acquiring_institution")
    .withColumnRenamed("Closing Date�", "closing_date")
    .withColumnRenamed("Fund", "fund")
)

In [9]:
# Remove o caractere � do conteúdo
for c in df_clean.columns:
    df_clean = df_clean.withColumn(c, regexp_replace(col(c), "�", ""))

df_clean.show(5)
df_clean.printSchema()

+--------------------+------------+-----+-----+---------------------+------------+-----+
|           bank_name|        city|state| cert|acquiring_institution|closing_date| fund|
+--------------------+------------+-----+-----+---------------------+------------+-----+
|The Santa Anna Na...|  Santa Anna|   TX| 5520| Coleman County St...|   27-Jun-25|10549|
|Pulaski Savings Bank|     Chicago|   IL|28611|      Millennium Bank|   17-Jan-25|10548|
|First National Ba...|     Lindsay|   OK| 4134| First Bank & Trus...|   18-Oct-24|10547|
|Republic First Ba...|Philadelphia|   PA|27332| Fulton Bank, Nati...|   26-Apr-24|10546|
|       Citizens Bank|    Sac City|   IA| 8758| Iowa Trust & Savi...|    3-Nov-23|10545|
+--------------------+------------+-----+-----+---------------------+------------+-----+
only showing top 5 rows

root
 |-- bank_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- cert: string (nullable = true)
 |-- acquiring_inst

In [13]:
output_path = r"C:\Users\00157NLUC-BrenoR\pos_data_analytics\banklist_clean"
df_clean.write.mode("overwrite").option("header", True).csv(output_path)

In [14]:
df_clean.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

## Using SQL in PySpark

In [11]:
df_clean.createOrReplaceTempView("banklist")

df_check = spark.sql("""select `Bank Name`, City, `Closing Date` from banklist""")
df_check.show(4, truncate=False)


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `Bank Name` cannot be resolved. Did you mean one of the following? [`bank_name`, `state`, `fund`, `cert`, `city`].; line 1 pos 7;
'Project ['Bank Name, City#237, 'Closing Date]
+- SubqueryAlias banklist
   +- View (`banklist`, [bank_name#229,city#237,state#245,cert#253,acquiring_institution#261,closing_date#269,fund#277])
      +- Project [bank_name#229, city#237, state#245, cert#253, acquiring_institution#261, closing_date#269, regexp_replace(fund#221, �, , 1) AS fund#277]
         +- Project [bank_name#229, city#237, state#245, cert#253, acquiring_institution#261, regexp_replace(closing_date#213, �, , 1) AS closing_date#269, fund#221]
            +- Project [bank_name#229, city#237, state#245, cert#253, regexp_replace(acquiring_institution#205, �, , 1) AS acquiring_institution#261, closing_date#213, fund#221]
               +- Project [bank_name#229, city#237, state#245, regexp_replace(cer

exercises in documentation

In [ ]:
import requests

path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'owid-covid-data.csv'
csv_file = open(csv_file_name, 'wb')

csv_file.write(url_content)
csv_file.close()

df = spark.read.csv(csv_file_name, header=True, inferSchema=True)


In [ ]:
#Viewing the dataframe schema 
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [ ]:
#Converting a date column
df.select(F.to_date(df.date).alias('date'))

DataFrame[date: date]

In [ ]:
#Summary stats
df.describe().show()

+-------+--------+-------------+-----------+-------------------+-----------------+------------------+------------------+------------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+-----------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+------------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+--------------------+--------------------+-----------------------+--------------------+-----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-----------------

In [ ]:
#Simple Group by Function
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(truncate=False)

+-----------------------------+--------------+
|location                     |sum(new_cases)|
+-----------------------------+--------------+
|World                        |775935057     |
|High-income countries        |429044052     |
|Asia                         |301564180     |
|Europe                       |252916868     |
|Upper-middle-income countries|251756125     |
|European Union (27)          |185822587     |
|North America                |124492698     |
|United States                |103436829     |
|China                        |99373219      |
|Lower-middle-income countries|92019711      |
|South America                |68811012      |
|India                        |45041748      |
|France                       |38997490      |
|Germany                      |38437756      |
|Brazil                       |37511921      |
|South Korea                  |34571873      |
|Japan                        |33803572      |
|Italy                        |26781078      |
|United Kingd